# 01 — Train a base model

Basic guide for training a base model. 
The original repository contains a CLI commands for usage and training. However we introduced and built a few more classes to aid in training and usage of the model, so that it is more user friendly but also customisable. 

In general there are a few key classes that are important for basic usage 
1) Dataloader (converts ASE atoms objects to pytorch dataloaders)
2) Trainer (settles the training, weight decay, checkpointing etc)
3) Model factory (function that helps to initialise model, with foundation weights)
4) Tester (tests the model based on various metrics)

There is also a key Strategies class, but that is covered separately in the second tutorial notebook, mainly used to customise the model specifically for transfer learning. 

Note that in this notebook the cells won't actually compile since the data filepaths are dummy file paths, not actual valid ones. But they contain valid syntax that will run if specified properly. 

## 1. Imports and basic settings

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import ase.io 

from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.modules.loss import InvariantsWeightedEnergyForcesNacsDipoleLoss
from mace.training.model_factory import initialise_autoencoder
from mace.training.trainer import Trainer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_default_dtype(torch.float64)

device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Read training and validation data

We start with an input of extended xyz files. This file will be read by ASE to form ASE atoms objects. This objects are used as the input for the dataloader later on 

For each geometry's xyz `atoms.info`, the following info is required: 

- `REF_energy`: `[states]`.
- `REF_forces`: `[atoms, states, 3]`.
- `REF_nacs`: `[atoms, pairs, 3]`, containing **raw NACs**.

NAC pairs are ordered `(0,1), (0,2), (1,2)`
Do **NOT** specify smooth NACs in the xyz file, that conversion is handeled internally. 

Units expected are Angstrom for distance and eV for energies for the xyz file. (By extension eV/A for forces, A-1 for raw NACs)

In [ ]:
data_file_path = "<insert training file path>"
valid_file_path = "<insert validation file path>"

train_atoms = ase.io.read(data_file_path, index=":")
valid_atoms = ase.io.read(valid_file_path, index=":")

## 3. Create the data loader builder

First we define a class called `AtomDataLoaderBuilder`. This builder is responsible for converting ASE atoms objects into the pytorch data loaders. Internally, it handles the graph conversions, labels readings, baseline energy levels, metadata, smooth/raw NAC conversions etc. 
The builder is initialised with parameters related to graph construction, and returns a dataloader when called with `load()` in the next step.

`cutoff=5.0` sets the neighbour distance limit in the same length units as the positions. The three label keys tell the builder where to find energies, forces, and raw NACs in `atoms.info`. These keys should match whatever is inside the xyz file

`E0s` can be specified manually as a dictionary of {element: energies}. If left unspecified as `None`, then it will be calculated based on the averages from the dataset. 

Note: For multiheaded training, E0s can be specified as a dictionary of dictionaries also (more details covered in tutorial 3 notebook)

In [ ]:
data_builder = AtomDataLoaderBuilder(
    cutoff=5.0, 
    energy_key="REF_energy", 
    forces_key="REF_forces", 
    nacs_key="REF_nacs",
    E0s= {"H": -15.357929595328724, "C": -1032.083979117871},
)


## 4. Build loaders and obtain metadata

When using the data builder class, we can return a pytorch dataloader object with the `load()`function. Here, we input the ASE atoms object from before, the desired batch size, and a dataloader is returned. 

`batch_size=10` refers to up to ten complete geometries per batch, not ten atoms. Training data is shuffled using our seed; validation data is not shuffled. 

`get_metadata()` returns the dataset information needed by the model: element identities (`atomic_numbers`), E0s (`atomic_energies`), cutoff (`r_max`), state count (`n_energies`), and neighbour statistics (`avg_num_neighbors`). 

This metadata is super important because its also used to initialise the model. Between train/validation, the same data builder should be used so that they share the same information (eg baseline energies). 

In [ ]:
train_loader = data_builder.load(
    train_atoms, 
    batch_size=10, 
    shuffle=True, 
    seed=seed
)

valid_loader = data_builder.load(
    valid_atoms, 
    batch_size=10, 
    shuffle=False
)

metadata = data_builder.get_metadata()

## 5. Initialise a fresh model

Now we need to initialise the model. The model should be initialised after the dataloaders because it requires metadata information like the num_elements, r_max, z_table etc. All these are handeled internally.

The dimensions of the model are controlled with the `preset` argument, which for not set as `"default_ani"`. This means that the number of layers etc will match the ani500k foundation model. 

`energy_decoder` can be either set to `"matrix"` for the original X-MACE implementation or `"direct"` to only predict `n_energies` without diagonalising the matrix. 

`load_base` can be set to `"ani500k"` which will point to the filepath of the ANI foundational model. This model is trained on CHNO compounds with CC level of theory. Alternatively you can manually specify a file path here as long as it points to another valid foundation model. 

If you like to play around with different layers etc, we can also manually specify the dimensions here (it will override the defaults). Eg if instead of 16 latent dims, you can chose 8 and it will override the default settings. This is fine because the latent dims is not part of the foundation model weights (current ones only cover the graph weights). However, if you change the weights and they have a mismatch with the foundation model, it will throw error. Use load_base=None if you want to train a fully freely customised architecture without foundation model. 


In [ ]:
model = initialise_autoencoder(
    metadata, 
    preset="default_ani", 
    latent_dim=16,
    energy_decoder="matrix", 
    compute_nacs=True, 
    load_base="ani500k"
)

## 6. Define the loss

Here we specify the loss function, for now we set the weights to all 1. Have not experimented much with this yet, but these will affect which loss terms are penalised more heavily. 

In [ ]:
loss_fn = InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, 
    forces_weight=1.0, 
    nacs_weight=1.0,
).to(device)

## 7. Configure Trainer

Now we configure the `Trainer` class. This class is used for training the model. The Trainer is initialised with all the training hyperparamters like the number of epochs, optimiser_LR etc etc. 

Internally, this class handels all the early stopping, EMA, weight decays, training, validation, checkpointing behaviour etc. 

In [ ]:
trainer = Trainer(
    device=device, 
    max_epochs=100, 
    optimiser_lr=1e-3,
    ema_decay=0.99, 
    restore_best=True
)

## 8. Choose an output directory

Here we specify a dummy output dir just as an example. 
The output will be used for checkpointing and saving models

In [ ]:
output_dir = Path("<insert new output directory>")
output_dir.mkdir(parents=True, exist_ok=False)

## 9. Train the model

To run the training, we call the function `train_model()` from the Trainer. 
We need to specify both the training and validation dataloaders that we previously obtained from the data_builder, as well as the loss function. 

For checkpointing, we specify the output directory for the checkpoints and the frequency that each checkpoint will be saved. 

This function will return a `model` and `history`, with the `model` being the best epoch model (based on lowest validation loss), as well as the training history, which contains all the relevant information like learning rate, training loss, validation loss, validation MAEs across epochs. This history is especially useful to save and debug. The losses are also split into individaul components (ie energy / forces / nacs all recorded separately). 

In [ ]:
model, history = trainer.train_model(
    model, 
    train_loader, 
    valid_loader, 
    loss_fn,
    checkpoint_epoch=10, 
    checkpoint_models_dir=output_dir / "checkpoints",
    compute_nacs=True
)

torch.save(model.state_dict(), output_dir / "best_model.pt")
(output_dir / "history.json").write_text(json.dumps(history, indent=2))

## 10. Evaluate with Tester

`Tester` runs the trained model on labelled test data and stores predictions and reference values. Use a separate test set to assess the model after training. Reuse `data_builder` so the test loader uses the training element order and E0s.

Call `run_test()` before requesting metrics. `compute_nacs=True` collects both raw and smooth NACs. The methods below report mean absolute errors (MAEs); NAC errors use consistent phase alignment within each geometry.

In [ ]:
from mace.testing import Tester

test_file_path = "<insert test file path>"
test_atoms = ase.io.read(test_file_path, index=":")
test_loader = data_builder.load(test_atoms, batch_size=10, shuffle=False)

tester = Tester(device=device)
tester.run_test(model, test_loader, compute_nacs=True)

print("Energy MAE:", tester.get_energy_mae())
print("Force MAE:", tester.get_force_mae())
print("Smooth NAC MAE:", tester.get_smooth_nac_phase_mae())
print("Raw NAC MAE:", tester.get_raw_nac_phase_mae())